In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [2]:
import pandas as pd
data = pd.read_csv("recommendations.csv")
data.head(1)

,Unnamed: 0,Respondent,advertiserurl,company,employmenttype_jobstatus,jobdescription,jobid,joblocation_address,jobtitle,postdate,shift,site_name,skills,uniq_id
0,0,1.0,https://www.dice.com/jobs/detail/Full-Stack-De...,Saicon Consultants Inc.,"Contract W2, 12 Months",Proficient in RESTful web servicesProficient i...,Dice Id : saicon,"Pleasanton, CA",Full Stack Developer,10 hours ago,Telecommuting not available|Travel not required,NaN,Full Stack Developer,90d755eb5b34547c31978ce7a6eb519f


In [ ]:
import pandas as pd
import random
from sentence_transformers import InputExample
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

df = pd.read_csv("recommendations.csv")
print(f"Loaded {len(df)} job records")

def create_contrastive_pairs(job_data, n_pairs=10000):
    
    examples = []
    
    HIGH_SIM = 0.9  # Components from same job
    LOW_SIM = 0.2   # Components from different jobs
    
    print("Creating positive pairs (components from same job)...")
    for _, row in job_data.iterrows():
        if pd.isna(row['jobtitle']) or pd.isna(row['jobdescription']) or pd.isna(row['skills']):
            continue
        
        examples.append(InputExample(
            texts=[row['jobtitle'], row['jobdescription']],
            label=HIGH_SIM
        ))
        
        examples.append(InputExample(
            texts=[row['jobdescription'], row['skills']],
            label=HIGH_SIM
        ))
        
        examples.append(InputExample(
            texts=[row['jobtitle'], row['skills']],
            label=HIGH_SIM
        ))
    
    print("Creating negative pairs (components from different jobs)...")
    
    n_negative = min(len(job_data) * 3, n_pairs // 2)
    
    for _ in range(n_negative):
        i, j = random.sample(range(len(job_data)), 2)
        
        if (pd.isna(job_data.iloc[i]['jobtitle']) or 
            pd.isna(job_data.iloc[i]['jobdescription']) or
            pd.isna(job_data.iloc[j]['skills'])):
            continue
        
        examples.append(InputExample(
            texts=[job_data.iloc[i]['jobtitle'], job_data.iloc[j]['skills']],
            label=LOW_SIM
        ))
        
        examples.append(InputExample(
            texts=[job_data.iloc[i]['jobdescription'], job_data.iloc[j]['skills']],
            label=LOW_SIM
        ))
    
    return examples

examples = create_contrastive_pairs(df)
train_examples, val_examples = train_test_split(examples, test_size=0.1, random_state=42)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
val_dataloader = DataLoader(val_examples, shuffle=False, batch_size=16)

Loaded 2000 job records
Creating positive pairs (components from same job)...
Creating negative pairs (components from different jobs)...

Created 16000 total examples
- Training examples: 14400
- Validation examples: 1600

Examples are ready for fine-tuning a sentence transformer model
You can use these examples with a loss function like CosineSimilarityLoss

Sample positive pairs (3):

Pair 1 (similarity: 0.9):
Text 1: Full Stack Developer...
Text 2: Proficient in RESTful web servicesProficient in micro service architectureExcellent in creating resp...

Pair 2 (similarity: 0.9):
Text 1: Proficient in RESTful web servicesProficient in micro service architectureExcellent in creating resp...
Text 2: Full Stack Developer...

Pair 3 (similarity: 0.9):
Text 1: Full Stack Developer...
Text 2: Full Stack Developer...

Sample negative pairs (3):

Pair 1 (similarity: 0.2):
Text 1: Node.js Developer...
Text 2: Java, NodeJS, Ruby, Scala, Python, Powershell, Bash, Azure, AWS, Rackspace, OpenStack

In [98]:
from sentence_transformers import SentenceTransformer, losses , evaluation

model = SentenceTransformer('all-MiniLM-L6-v2')
train_loss = losses.CosineSimilarityLoss(model)

evaluator = evaluation.EmbeddingSimilarityEvaluator.from_input_examples(
    val_examples,  # This uses the validation examples directly
    name='job-validation'
)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,  # Add the evaluator here
    evaluation_steps=1000,  # Evaluate every 1000 steps
    epochs=10,
    warmup_steps=100,
    output_path='./job_embeddings_model_evaluated2',
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Job-validation Pearson Cosine,Job-validation Spearman Cosine
900,0.057100,No log,0.823221,0.779110
1000,0.039400,No log,0.827456,0.781855
1800,0.031000,No log,0.864310,0.801253
2000,0.028100,No log,0.866130,0.802242
2700,0.024300,No log,0.888026,0.810346
3000,0.020600,No log,0.891517,0.811092
3600,0.019000,No log,0.902554,0.815113
4000,0.016800,No log,0.905901,0.815087
4500,0.016000,No log,0.909917,0.817646
5000,0.013900,No log,0.913899,0.817687


In [ ]:
from sentence_transformers import SentenceTransformer

model2 = model = SentenceTransformer('./job_embeddings_model_evaluated2')
print("Model2 loaded successfully!")

d:\anaconda\envs\rec\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model loaded successfully!
Model2 loaded successfully!


In [8]:
import pandas as pd
data = pd.read_csv("E:/vs codes/REC/data/job_lists_export.csv" , delimiter=';')
data.head(1)

,id,title,user_id,location,date_posted,description,skills,min_salary,max_salary,salary_negotiable,payment_period,payment_currency,hiring_multiple_candidates
0,1,"Application Developer - ERP, Commercial CPS",12287,"Atlanta, GA, US",2025-04-15,Cargill’s size scale allows us make positive i...,"acceptance testing,application,application sup...",71538,91833,1,monthly,USD,0


In [9]:
data['full_text'] = data['title'] + ' ' + data['description']
data.head(1)

,id,title,user_id,location,date_posted,description,skills,min_salary,max_salary,salary_negotiable,payment_period,payment_currency,hiring_multiple_candidates,full_text
0,1,"Application Developer - ERP, Commercial CPS",12287,"Atlanta, GA, US",2025-04-15,Cargill’s size scale allows us make positive i...,"acceptance testing,application,application sup...",71538,91833,1,monthly,USD,0,"Application Developer - ERP, Commercial CPS Ca..."


In [7]:
job_embeddings = model.encode(df['full_text'].tolist(), show_progress_bar=True)
job_embeddings2 = model2.encode(df['full_text'].tolist(), show_progress_bar=True)

Batches: 100%|██████████| 590/590 [04:47<00:00,  2.05it/s]


In [ ]:
#import cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity
def recommend_jobs(query_text, top_k=5):
    # Encode the query into an embedding vector
    query_embedding = model2.encode(query_text)
    
    # Calculate similarity between query and all jobs
    similarities = cosine_similarity([query_embedding], job_embeddings2)[0]
    
    # Find indices of top-k most similar jobs
    top_indices = similarities.argsort()[::-1][:top_k]
    
    # Create a results dataframe with top-k jobs
    results = df.iloc[top_indices].copy()
    
    # Add similarity scores and format as percentages
    results['similarity'] = similarities[top_indices]
    results['match_percentage'] = results['similarity'].apply(lambda x: f"{x*100:.1f}%")
    results['idx'] = top_indices
    
    # Return relevant columns
    return results[["idx" , "title", 'match_percentage']]

# Example usage
query = "iam an expert in backend with laraval skills , created multiple projects using laravel and php frameworks"
recommendations2 = recommend_jobs(query)

In [11]:
recommendations2

,idx,title,match_percentage
25,25,Laravel / PHP Developer,84.4%
18866,18866,Senior Laravel Developer,83.6%
18865,18865,Senior Laravel Developer,83.6%
14243,14243,Laravel PHP Technical Support Dev,78.9%
3236,3236,PHP Developer – strong DevOps/Laravel – Urgent,77.4%


In [ ]:
user1 = "iam a junior backend developer with basic larval skills and small experience in php"
recommendations2 = recommend_jobs(query)
recommendations2

,idx,title,match_percentage
6791,6791,Remote Full Stack Web Developer - PHP/CSS/MySQL,86.2%
91,91,Pure Php Developer,86.2%
85,85,Junior Level PHP Developer,85.9%
86,86,Senior Level PHP Developer,85.7%
6677,6677,Full Stack ERP Developer (New Jersey Based - N...,85.5%


In [4]:
import pandas as pd
data = pd.read_csv("job_lists_export.csv" , delimiter=";")  # Read in the job listings CSV file
data.head(1)

,id,title,user_id,location,date_posted,description,skills,min_salary,max_salary,salary_negotiable,payment_period,payment_currency,hiring_multiple_candidates
0,1,"Application Developer - ERP, Commercial CPS",12287,"Atlanta, GA, US",2025-04-15,Cargill’s size scale allows us make positive i...,"acceptance testing,application,application sup...",71538,91833,1,monthly,USD,0
